# Image Preprocessing and Facial Recognition

This notebook loads the team photos from `datasets/images/<identity>/`, handles JPEG and HEIC files, detects and crops faces, demonstrates image augmentation, exports HOG features to `datasets/image_features.csv`, evaluates a classifier on one held-out original image per identity, and saves the final model to `models/face_model.pkl`.

The current dataset is small (three original photos per person), so the evaluation is a demonstration rather than evidence of production-ready identity verification.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from facial_recognition import (
    IMAGE_DIR,
    FEATURES_PATH,
    MODEL_PATH,
    augment_face_images,
    discover_images,
    load_face_crop,
    train_face_model,
    verify_face,
)

## Review the collected images

Identity labels come from the parent folder names, ensuring consistent labels across each member's photos.

In [ ]:
image_paths = discover_images(IMAGE_DIR)
inventory = pd.DataFrame(
    [{"identity": identity, "image": path.name, "format": path.suffix.lower()} for identity, path in image_paths]
)
display(inventory.groupby("identity").size().rename("image_count").to_frame())
display(inventory)

## Face crops and augmentations

The pipeline applies rotation, horizontal flipping, grayscale conversion, and brightness adjustment to each detected face crop. HOG (Histogram of Oriented Gradients) descriptors represent local facial shape and texture.

In [ ]:
identity, sample_path = image_paths[0]
face_crop = load_face_crop(sample_path)
augmentations = augment_face_images(face_crop)

fig, axes = plt.subplots(1, len(augmentations), figsize=(15, 3))
for axis, (name, image) in zip(axes, augmentations.items()):
    axis.imshow(image)
    axis.set_title(name)
    axis.axis("off")
fig.suptitle(f"{identity}: sample face crop and augmentations")
plt.tight_layout()

## Extract features, evaluate, and train the final model

Evaluation holds out one original image for each identity. Augmented versions of those held-out images are excluded from evaluation training to avoid data leakage. After reporting metrics, the saved model is retrained using the full set of originals and their augmentations.

In [ ]:
metrics = train_face_model()
print("Evaluation metrics:")
display(pd.Series(metrics, name="score").to_frame())
print(f"Feature table: {FEATURES_PATH}")
print(f"Saved model: {MODEL_PATH}")

## Verify a face

The classifier returns an identity only when its top class probability meets the configured threshold. Add an authorized test photo path below to try another input. An unknown-person rejection test requires a separate, consented photo that is not part of the four-member training set.

In [ ]:
test_identity, test_path = image_paths[0]
print(f"Known sample ({test_identity}):", verify_face(test_path))

unauthorized_path = ROOT / "datasets" / "images" / "unauthorized" / "test_person.jpg"
if unauthorized_path.is_file():
    print("Separate unknown-person sample:", verify_face(unauthorized_path))
else:
    print("No unauthorized test image supplied; no unknown-person result is claimed.")

## Limitations

Only three original images are available for each person. Augmentation increases training variety but does not replace collecting more independent images. This HOG/SVM baseline is for the assignment demonstration, not high-security authentication; it does not provide liveness or spoof detection.